<a href="https://colab.research.google.com/github/jovanfranco/usdmxn-volatilidad/blob/main/clase23_momentum_fx.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clase 23: momentum (tendencia) en divisas

Señal de tendencia por par, escalada por volatilidad, con costos y sin carry. Ventanas fijadas antes de ver resultados.

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
from scipy.stats import norm

# Ejemplo a mano: un par, ventana de 3 dias. Señal en el cierre de t = signo(precio_t / precio_(t-3) - 1)
# La posicion se toma al cierre de t y gana el rendimiento de t+1 (nunca el mismo dia: eso seria hacer trampa)
precios = pd.Series([100, 102, 104, 103, 101, 100, 102], dtype=float)
L = 3
r = precios.pct_change()
senal = np.sign(precios / precios.shift(L) - 1)
pnl = senal.shift(1) * r          # la señal de ayer gana el rendimiento de hoy
tabla = pd.DataFrame({"precio": precios, "rend_%": (r * 100).round(3), "señal_al_cierre": senal, "pnl_%": (pnl * 100).round(3)})
print(tabla.to_string())
print("PnL total (sin escala de volatilidad): %.3f%%" % (pnl.sum() * 100))

# Escala por volatilidad: posicion = señal * (vol objetivo / vol del activo)
for vol_activo in (0.10, 0.20):
    print("Vol del activo", vol_activo, "-> tamaño de la posicion:", 0.10 / vol_activo)


   precio  rend_%  señal_al_cierre  pnl_%
0   100.0     NaN              NaN    NaN
1   102.0   2.000              NaN    NaN
2   104.0   1.961              NaN    NaN
3   103.0  -0.962              1.0    NaN
4   101.0  -1.942             -1.0 -1.942
5   100.0  -0.990             -1.0  0.990
6   102.0   2.000             -1.0 -2.000
PnL total (sin escala de volatilidad): -2.952%
Vol del activo 0.1 -> tamaño de la posicion: 1.0
Vol del activo 0.2 -> tamaño de la posicion: 0.5


In [2]:
# Funciones
def vol_ewma(rend, lam=0.94):
    """Volatilidad anualizada con EWMA, usando datos hasta t (inclusive)."""
    return np.sqrt((rend ** 2).ewm(alpha=1 - lam, adjust=False).mean() * 252)

def posiciones_mom(rend, L, vol_objetivo=0.10, tope=3.0):
    """Posicion por par al cierre de t: signo del rendimiento de las ultimas L dias, escalada por volatilidad."""
    acum = (1 + rend).rolling(L).apply(np.prod, raw=True) - 1
    senal = np.sign(acum)
    escala = (vol_objetivo / vol_ewma(rend)).clip(upper=tope)
    return senal * escala

def posiciones_combo(rend, Ls, vol_objetivo=0.10, tope=3.0):
    """Promedio de las señales de varias ventanas: queda entre menos 1 y 1."""
    senales = [np.sign((1 + rend).rolling(L).apply(np.prod, raw=True) - 1) for L in Ls]
    escala = (vol_objetivo / vol_ewma(rend)).clip(upper=tope)
    return sum(senales) / len(Ls) * escala

def backtest(rend, pos, costo_bps=0.0):
    """pos al cierre de t gana el rendimiento de t+1. La cartera es el promedio de los pares.
    El costo se cobra por unidad operada: |cambio de posicion| por costo_bps."""
    pos_usada = pos.shift(1)
    bruto = (pos_usada * rend).mean(axis=1)
    rotacion = pos_usada.diff().abs().mean(axis=1)
    rotacion = rotacion.fillna(0.0)
    return bruto - rotacion * costo_bps / 10000, rotacion

def metricas(pnl):
    pnl = pnl.dropna()
    anios = len(pnl) / 252
    sharpe = pnl.mean() / pnl.std() * np.sqrt(252)
    curva = (1 + pnl).cumprod()
    dd = (curva / curva.cummax() - 1).min()
    t = sharpe * np.sqrt(anios)
    p = 2 * (1 - norm.cdf(abs(t)))
    return {"rend_%": pnl.mean() * 252 * 100, "vol_%": pnl.std() * np.sqrt(252) * 100,
            "Sharpe": sharpe, "t": t, "p": p, "maxDD_%": dd * 100}

# Verificacion con el ejemplo a mano
chequeo = backtest(pd.DataFrame({"x": precios.pct_change()}), pd.DataFrame({"x": np.sign(precios / precios.shift(3) - 1)}))[0]
print("Chequeo del ejemplo a mano (debe ser menos 2.952%):", round(chequeo.sum() * 100, 3))


Chequeo del ejemplo a mano (debe ser menos 2.952%): -2.952


In [3]:
# Datos reales: seis pares contra el dolar (solo precio, sin carry)
pares = {"USD/MXN": ("MXN=X", False), "USD/JPY": ("JPY=X", False), "USD/EUR": ("EURUSD=X", True),
         "USD/GBP": ("GBPUSD=X", True), "USD/CAD": ("CAD=X", False), "USD/BRL": ("BRL=X", False)}
px = {}
for nombre, (ticker, invertir) in pares.items():
    p = yf.download(ticker, start="2020-10-07", end="2026-10-22", progress=False, auto_adjust=True)["Close"].squeeze()
    px[nombre] = 1 / p if invertir else p
rend = pd.DataFrame(px).dropna().pct_change().dropna()
print("Dias:", len(rend), "| del", rend.index[0].date(), "al", rend.index[-1].date())


Dias: 1570 | del 2020-10-08 al 2026-10-21


In [4]:
# Ventanas elegidas ANTES de ver resultados: 1, 3, 6 y 12 meses. Se reportan todas, sin escoger la mejor.
LS = [21, 63, 126, 252]
INICIO = rend.index[max(LS) + 1]       # todas se comparan en las mismas fechas
print("Evaluacion desde:", INICIO.date(), "(los primeros", max(LS), "dias se usan para calentar la señal)")

resultados, pnls = {}, {}
for L in LS:
    pos = posiciones_mom(rend, L)
    pnls[f"Momentum {L}d"] = backtest(rend, pos)[0].loc[INICIO:]
pos_combo = posiciones_combo(rend, LS)
pnls["Combo (promedio de las 4)"] = backtest(rend, pos_combo)[0].loc[INICIO:]
siempre_usd = (vol_ewma(rend).rdiv(0.10)).clip(upper=3.0) * 1.0     # largo USD en todos los pares, misma escala de vol
pnls["Siempre largo USD"] = backtest(rend, siempre_usd)[0].loc[INICIO:]

tabla = pd.DataFrame({k: metricas(v) for k, v in pnls.items()}).T
n_pruebas = len(LS) + 1
tabla["p_Bonferroni"] = (tabla["p"] * n_pruebas).clip(upper=1.0)
tabla.loc["Siempre largo USD", "p_Bonferroni"] = np.nan
print(tabla.round(3).to_string())
anios = len(pnls["Momentum 21d"]) / 252
print("Años evaluados:", round(anios, 2), "| error estandar aproximado del Sharpe:", round(1 / np.sqrt(anios), 2))
print("Bonferroni con", n_pruebas, "pruebas (las señales estan correlacionadas, asi que es conservador).")


Evaluacion desde: 2021-09-28 (los primeros 252 dias se usan para calentar la señal)
                           rend_%  vol_%  Sharpe      t      p  maxDD_%  p_Bonferroni
Momentum 21d               -1.979  5.994  -0.330 -0.755  0.450  -14.947           1.0
Momentum 63d                0.797  5.815   0.137  0.313  0.754  -10.360           1.0
Momentum 126d              -1.266  5.744  -0.220 -0.504  0.614  -21.989           1.0
Momentum 252d               1.084  5.223   0.207  0.474  0.635   -9.184           1.0
Combo (promedio de las 4)  -0.341  4.231  -0.081 -0.184  0.854  -10.370           1.0
Siempre largo USD           3.763  7.504   0.501  1.146  0.252  -11.749           NaN
Años evaluados: 5.23 | error estandar aproximado del Sharpe: 0.44
Bonferroni con 5 pruebas (las señales estan correlacionadas, asi que es conservador).


In [5]:
# Estabilidad: ¿el resultado es de todo el periodo o de un pedazo?
mitad = len(pnls["Combo (promedio de las 4)"]) // 2
filas = []
for k, v in pnls.items():
    a, b = v.iloc[:mitad], v.iloc[mitad:]
    filas.append({"estrategia": k, "Sharpe 1a mitad": a.mean() / a.std() * np.sqrt(252), "Sharpe 2a mitad": b.mean() / b.std() * np.sqrt(252)})
print(pd.DataFrame(filas).round(2).to_string(index=False))
print("\nSharpe del Combo por año calendario:")
c = pnls["Combo (promedio de las 4)"]
print(c.groupby(c.index.year).apply(lambda s: s.mean() / s.std() * np.sqrt(252) if len(s) > 20 else np.nan).round(2).to_string())

print("\nFraccion de dias largo USD por par (señal de 126 dias):")
print((np.sign((1 + rend).rolling(126).apply(np.prod, raw=True) - 1).loc[INICIO:] > 0).mean().round(2).to_string())
print("Correlacion del Combo con 'Siempre largo USD':", round(pnls["Combo (promedio de las 4)"].corr(pnls["Siempre largo USD"]), 2))


               estrategia  Sharpe 1a mitad  Sharpe 2a mitad
             Momentum 21d            -0.30            -0.36
             Momentum 63d             0.29            -0.01
            Momentum 126d             0.38            -0.82
            Momentum 252d             0.60            -0.16
Combo (promedio de las 4)             0.31            -0.45
        Siempre largo USD             0.67             0.34

Sharpe del Combo por año calendario:
Date
2021    0.44
2022    0.95
2023    0.11
2024    0.23
2025   -0.83
2026   -1.31

Fraccion de dias largo USD por par (señal de 126 dias):
USD/MXN    0.35
USD/JPY    0.70
USD/EUR    0.52
USD/GBP    0.51
USD/CAD    0.67
USD/BRL    0.45
Correlacion del Combo con 'Siempre largo USD': 0.12


In [6]:
# Costos: el costo por unidad operada (en puntos base) es un SUPUESTO mio, no un dato de tu mesa. Cambialo por el real.
print("Rotacion anual (unidades de posicion operadas por año, promedio de los pares):")
filas = []
for nombre, pos in [("Momentum 21d", posiciones_mom(rend, 21)), ("Momentum 63d", posiciones_mom(rend, 63)),
                    ("Momentum 252d", posiciones_mom(rend, 252)), ("Combo (promedio de las 4)", pos_combo)]:
    fila = {"estrategia": nombre}
    rot = backtest(rend, pos)[1].loc[INICIO:]
    fila["rotacion_anual"] = rot.mean() * 252
    for bps in (0, 1, 2, 5, 10):
        pn = backtest(rend, pos, bps)[0].loc[INICIO:]
        fila[f"Sharpe {bps}bps"] = pn.mean() / pn.std() * np.sqrt(252)
    filas.append(fila)
print(pd.DataFrame(filas).round(2).to_string(index=False))


Rotacion anual (unidades de posicion operadas por año, promedio de los pares):
               estrategia  rotacion_anual  Sharpe 0bps  Sharpe 1bps  Sharpe 2bps  Sharpe 5bps  Sharpe 10bps
             Momentum 21d           71.40        -0.33        -0.45        -0.57        -0.92         -1.51
             Momentum 63d           48.64         0.14         0.05        -0.03        -0.28         -0.70
            Momentum 252d           34.83         0.21         0.14         0.07        -0.13         -0.46
Combo (promedio de las 4)           43.37        -0.08        -0.18        -0.29        -0.59         -1.10
